In [ ]:
## In this example, we will study output data frame from pandora.py configuration
#### 1. Opening each data frame and check structure
#### 2. Collect POT and scale factor to the target POT
#### 3. Merge evtdf and mcnudf for further study
#### 4. Draw some plots for each slice and for each pfp

from analysis_village.cc1pi.var_configs import *

import os
import sys
import lmfit
import numpy as np
import math
import uproot as uproot
import pickle
import pandas as pd
import gc

import matplotlib.pyplot as plt
import matplotlib.colors
from matplotlib.colors import LinearSegmentedColormap
from matplotlib import ticker
from matplotlib.ticker import (AutoMinorLocator, MultipleLocator)
from matplotlib import gridspec

from analysis_village.unfolding.wienersvd import *
from analysis_village.unfolding.unfolding_inputs import *

# Add the head direcoty to sys.path
workspace_root = os.getcwd()  
sys.path.insert(0, workspace_root + "/../../")

# Absolute path to cafpyana directory
print('Importing cafpyana utils...')
cafpyana_root = "/home/lpelegri/cafpyana"
# Add CAFpyana to the Python search path -- this will allow you to import cafpyana modules
sys.path.insert(0, cafpyana_root)


from analysis_village.cc1pi.HelperFunctions import HelperFunctions
from analysis_village.cc1pi.Constants import CTE as CTE
from analysis_village.cc1pi.CutMasks import CutMasks
from analysis_village.cc1pi.CutMasks.MaskUtils import *
from analysis_village.cc1pi.DataFrameUtils import DFCleaning
from analysis_village.cc1pi.DataFrameUtils.DFLoading import *
from analysis_village.cc1pi.GraphUtils.GraphsUtils import *
from analysis_village.cc1pi.GraphUtils.Utils import *

# import this repo's classes
import pyanalib.pandas_helpers as ph
import pyanalib.split_df_helpers as splh
import pyanalib.stat_helpers as sh

from pyanalib.split_df_helpers import *
from analysis_village.cc1pi.systematics.final_variable_configs import VariableConfig
from analysis_village.cc1pi.systematics.utils import *
from analysis_village.cc1pi.systematics.constants import *
from pyanalib.covariance import *
from analysis_village.cc1pi.DataFrameUtils.DFLoading import *

np.seterr(divide='ignore', invalid='ignore', over='ignore')

# Load DataFrame MC

In [ ]:
#Load CV dataframe
keys2load = ["cc1pi", "hdr", "histpotdf", "nudf"] ## keys from the configuration file
bnb_path = "/exp/sbnd/data/users/lpelegri/cafpyana_data/mc_ar23p_extended_syst_pruned.df"
mc_bnb_df = load_df(bnb_path, keys2load, 100)

mc_bnb_evt_df = mc_bnb_df['cc1pi']
mc_bnb_evt_df = mc_bnb_evt_df[build_event_cumulative_masks(mc_bnb_evt_df, sideband = "")["extra_pion"]]
mc_bnb_nu_df = mc_bnb_df['nudf']
mc_bnb_hdr_df = mc_bnb_df['hdr']
cols_to_keep = [
    ('nu_categ', '', '', ''),
    ('genie_categ', '', '', ''),
    ('nu_categ_proton_reduced', '', '', '')
]
mc_bnb_nu_df = mc_bnb_nu_df[cols_to_keep]

#Load data
keys2load = ["cc1pi", "hdr", "histpotdf"] ## keys from the configuration file
data_df = load_df("/exp/sbnd/data/users/lpelegri/cafpyana_data/cc1pi_data_rollingdev_bnblight.df", keys2load, 100)
data_evt_df = data_df['cc1pi']
data_evt_df = data_evt_df[build_event_cumulative_masks(data_evt_df, sideband = "")["extra_pion"]]
data_hdr_df = data_df['hdr']

In [ ]:
pot_weight_col = ('slc', 'wgt', '', '', '', '')
# BNB data
data_tot_pot = data_hdr_df['pot'].sum()
print("data_tot_pot: %.3e" %(data_tot_pot))
data_evt_df[pot_weight_col] = np.ones(len(data_evt_df))
data_gates = data_hdr_df.nbnbinfo.sum()
print("data tot gates : %.3e" %(data_gates))

# BNB MC
mc_tot_pot = mc_bnb_hdr_df['pot'].sum()
print("mc_tot_pot: %.3e" %(mc_tot_pot))
mc_pot_scale = data_tot_pot / mc_tot_pot
print("mc_pot_scale: %.3e" %(mc_pot_scale))
mc_bnb_evt_df[pot_weight_col] = mc_pot_scale * np.ones(len(mc_bnb_evt_df))

In [ ]:
mc_evt_df = mc_bnb_evt_df
if "ar23p" in bnb_path:
    mc_evt_df = perform_truth_matching_low_memmory(mc_evt_df, mc_bnb_nu_df)
else:
    mc_evt_df = perform_truth_matching(mc_evt_df, mc_bnb_nu_df)   
print("Finished loading")    

n_mc = get_n_evt(mc_bnb_evt_df, True)
n_data = get_n_evt(data_evt_df, False)

print(f"{n_mc:<12} | {n_data:<12}")

In [ ]:
from tqdm.auto import tqdm
import pandas as pd
import numpy as np

# Register tqdm with pandas
tqdm.pandas(desc="Calculating Muon/Pion Vars")

def add_mu_pi_vars_column(pandora_df):
    # Fixed the list syntax and typos (removed ':' and fixed trk_end columns)
    EXPECTED_COLS = [
        'TL_mu','TL_pi',
        'dir_mu_x', 'dir_mu_y', 'dir_mu_z',
        'dir_pi_x', 'dir_pi_y', 'dir_pi_z',
        'trk_end_mu_x', 'trk_end_mu_y', 'trk_end_mu_z',
        'trk_end_pi_x', 'trk_end_pi_y', 'trk_end_pi_z'
    ]
    
    group_levels = ['__ntuple', 'entry', 'rec.slc..index']
    
    # 1. Group and Apply
    muon_pion_vars = (
        pandora_df.groupby(level=group_levels, group_keys=False)
            .progress_apply(get_mu_pi_vars)
    )
    
    # Get all unique slice indices from the original DF
    all_slices_idx = pandora_df.index.droplevel('rec.slc.reco.pfp..index').unique()
    
    if muon_pion_vars.empty:
        slcdf = pd.DataFrame(index=all_slices_idx, columns=EXPECTED_COLS)
    else:
        # 2. Fix the duplicate index issue
        # This ensures we have one set of variables per slice
        if muon_pion_vars.index.duplicated().any():
            muon_pion_vars = muon_pion_vars[~muon_pion_vars.index.duplicated(keep='first')]

        if isinstance(muon_pion_vars, pd.Series):
            if not isinstance(muon_pion_vars.index, pd.MultiIndex):
                 muon_pion_vars = muon_pion_vars.to_frame().T
            else:
                 muon_pion_vars = muon_pion_vars.unstack()
                
        # Safe reindex for columns and then index
        slcdf = muon_pion_vars.reindex(columns=EXPECTED_COLS)
        slcdf = slcdf.reindex(all_slices_idx)
    
    # 3. Enforce schema
    for col in EXPECTED_COLS:
        slcdf[col] = pd.to_numeric(slcdf[col], errors='coerce').astype('float32')
            
    # 4. Apply MultiIndex columns
    slcdf.columns = pd.MultiIndex.from_tuples(
        [('slc', 'measure_var', col, '', '', '') for col in slcdf.columns]
    )
    
    # 5. Join back 
    # Use 'left' join on the index levels
    pandora_df = pandora_df.join(slcdf, on=group_levels)
    
    return pandora_df

def get_mu_pi_vars(group):
    # Safety check for empty or tiny groups
    if len(group) < 2:
        return pd.Series({col: -999.0 for col in [
            'dir_mu_x', 'dir_mu_y', 'dir_mu_z', 'dir_pi_x', 'dir_pi_y', 'dir_pi_z',
            'trk_end_mu_x', 'trk_end_mu_y', 'trk_end_mu_z', 'trk_end_pi_x', 'trk_end_pi_y', 'trk_end_pi_z', 
        'TL_mu', 'TL_pi'
        ]})

    # --- MUON SELECTION ---
    # Using try-except or check to handle potential CutMasks errors
    try:
        exiting_mask = CutMasks.exiting_pfp_mask(group)
    except:
        exiting_mask = np.zeros(len(group), dtype=bool)
    
    if exiting_mask.sum() >= 1:
        muon_row = group.loc[exiting_mask].iloc[[0]]
    else:
        # Sort by BDT score
        group_sorted = group.sort_values(('pfp','trk','bdt_muon_pion_score','','',''))
        muon_row = group_sorted.iloc[[-1]]
      
    # --- PION SELECTION ---
    remaining_pfps = group.drop(muon_row.index)
    pion_row = remaining_pfps.sort_values(('pfp','trk','len','','','')).iloc[[-1]]
    
    # Extract values
    return pd.Series({
        'TL_mu': muon_row.pfp.trk.len.iloc[0],
        'TL_pi': pion_row.pfp.trk.len.iloc[0],
        
        'dir_mu_x': muon_row.pfp.trk.dir.x.iloc[0],
        'dir_mu_y': muon_row.pfp.trk.dir.y.iloc[0],
        'dir_mu_z': muon_row.pfp.trk.dir.z.iloc[0],
        'dir_pi_x': pion_row.pfp.trk.dir.x.iloc[0],
        'dir_pi_y': pion_row.pfp.trk.dir.y.iloc[0],
        'dir_pi_z': pion_row.pfp.trk.dir.z.iloc[0],
        
        'trk_end_mu_x': muon_row.pfp.trk.end.x.iloc[0],
        'trk_end_mu_y': muon_row.pfp.trk.end.y.iloc[0],
        'trk_end_mu_z': muon_row.pfp.trk.end.z.iloc[0],
        
        'trk_end_pi_x': pion_row.pfp.trk.end.x.iloc[0],
        'trk_end_pi_y': pion_row.pfp.trk.end.y.iloc[0],
        'trk_end_pi_z': pion_row.pfp.trk.end.z.iloc[0],
    })


In [ ]:
# 1. Define the boolean mask for candidates
# We use data_evt_df instead of pandora_df here
candidate_mask = (
    data_evt_df.slc.cut.inside_FV & 
    data_evt_df.slc.cut.t0 & 
    data_evt_df.slc.cut.track & 
    CutMasks.is_MIP_candidate_mask(data_evt_df) & 
    data_evt_df.slc.cut.containment
)
# 3. Pass the candidate dataframe into your variable-adding function
# Note: Since the function returns the modified DF, we assign it back
data_evt_df = add_mu_pi_vars_column(data_evt_df[candidate_mask])

In [ ]:
candidate_mask = (
    mc_evt_df.slc.cut.inside_FV & 
    mc_evt_df.slc.cut.t0 & 
    mc_evt_df.slc.cut.track & 
    CutMasks.is_MIP_candidate_mask(mc_evt_df) & 
    mc_evt_df.slc.cut.containment
)
# 3. Pass the candidate dataframe into your variable-adding function
# Note: Since the function returns the modified DF, we assign it back
mc_evt_df = add_mu_pi_vars_column(mc_evt_df[candidate_mask])


In [ ]:
'''
x_col = ('slc', 'measure_var', 'dir_mu_x', '', '', '')
y_col = ('slc', 'measure_var', 'dir_mu_y', '', '', '')
phi_col = ('slc', 'measure_var', 'phi_mu', '', '', '')

# 2. Calculate using arctan2 (y, x)
# Note: typically phi is arctan2(y, x)
mc_evt_df[phi_col] = np.arctan(
    mc_evt_df[y_col]/
    mc_evt_df[x_col]
)

data_evt_df[phi_col] = np.arctan(
    data_evt_df[y_col]/
    data_evt_df[x_col]
)
'''
import numpy as np

# 1. Define the column keys
x_col = ('slc', 'measure_var', 'dir_mu_x', '', '', '')
y_col = ('slc', 'measure_var', 'dir_mu_y', '', '', '')
phi_col = ('slc', 'measure_var', 'phi_mu', '', '', '')

# Convert MC to degrees
mc_evt_df[phi_col] = np.degrees(np.arctan2(mc_evt_df[y_col], mc_evt_df[x_col]))

# Convert Data to degrees
data_evt_df[phi_col] = np.degrees(np.arctan2(data_evt_df[y_col], data_evt_df[x_col]))



# 1. Define the column keys
x_col = ('slc', 'measure_var', 'dir_pi_x', '', '', '')
y_col = ('slc', 'measure_var', 'dir_pi_y', '', '', '')
phi_col = ('slc', 'measure_var', 'phi_pi', '', '', '')

# Convert MC to degrees
mc_evt_df[phi_col] = np.degrees(np.arctan2(mc_evt_df[y_col], mc_evt_df[x_col]))

# Convert Data to degrees
data_evt_df[phi_col] = np.degrees(np.arctan2(data_evt_df[y_col], data_evt_df[x_col]))


In [ ]:

R_mc = np.sqrt(mc_evt_df[x_col]**2 + mc_evt_df[y_col]**2)
print(f"Average Transverse Magnitude: {R_mc.mean()}")

In [ ]:
group_levels =  ['__ntuple','entry', 'rec.slc..index']

In [ ]:
def TPC_containment_mask(df, group_levels):
    """
    Keeps only slices where ALL pfp track endpoints (start and end)
    have the same sign in x as the slice vertex x.
    """
    vertex_x = df[('slc','vertex','x','','','')] .groupby(level=group_levels).first()
    
    # Map vertex x sign back to every row
    vertex_sign = np.sign(vertex_x)
    row_vertex_sign = df.index.droplevel('rec.slc.reco.pfp..index').map(vertex_sign)
    
    # Check if either endpoint violates the sign condition
    start_bad = np.sign(df[('pfp','trk','start','x','','')]) != row_vertex_sign
    end_bad   = np.sign(df[('pfp','trk','end','x','','')])   != row_vertex_sign

    # A slice is invalid if ANY of its pfps violates the condition
    violating_df = df[start_bad | end_bad]
    violating_slices = violating_df.groupby(level=group_levels).size()
    invalid_slices = violating_slices[violating_slices > 0].index

    mask = pd.Series(
        ~df.index.droplevel('rec.slc.reco.pfp..index').isin(invalid_slices),
        index=df.index
    )
    return mask

In [ ]:
print(mc_evt_df.pfp.trk.len)
print(mc_evt_df.pfp.trk.len)

In [ ]:
n_mc = get_n_evt(mc_evt_df, True)
n_data = get_n_evt(data_evt_df, False)

print(f"{n_mc:<12} | {n_data:<12}")

n_mc_2 = get_n_evt(mc_evt_df[TPC_containment_mask(mc_evt_df,group_levels)], True)
n_data_2 = get_n_evt(data_evt_df[TPC_containment_mask(data_evt_df,group_levels)], False)

print(f"{n_mc_2:<12} | {n_data_2:<12}")
print(n_mc_2/n_mc)


HelperFunctions.print_purity(mc_evt_df, ('truth','nu_categ','','','',''))
HelperFunctions.print_purity(mc_evt_df[TPC_containment_mask(mc_evt_df,group_levels)], ('truth','nu_categ','','','',''))


In [ ]:


config_phi_mu = FullHistogramConfig(
    file_name = "",      
    var_evt_reco_col=('slc','measure_var','phi_mu','','',''),
    truth_column = nu_categ_column,
    first_per_slice = True,
    start_cut = "energy",
    end_cut = "energy",
    bins=np.linspace(-180, 180, 21),
    xlabel=r'phi_mu',
    ylabel=slices_y_label
)

config_trk_end_x = FullHistogramConfig(
    file_name = "",      
    var_evt_reco_col=('slc','measure_var','trk_end_mu_x','','',''),
    truth_column = nu_categ_column,
    first_per_slice = True,
    start_cut = "energy",
    end_cut = "energy",
    bins=np.linspace(-200, 200, 21),
    xlabel=r'trk end x [cm]',
    ylabel=slices_y_label
)


mask_dict = {
    "TPC0": lambda df: (df.slc.vertex.x < 0) & (df.slc.cut_var.n_exiting_pfps == 0),
    "TPC1": lambda df: (df.slc.vertex.x > 0) & (df.slc.cut_var.n_exiting_pfps == 0),
    "TPC_contained": lambda df: TPC_containment_mask(df,group_levels) & (df.slc.cut_var.n_exiting_pfps == 0),
}

var_configs = [config_phi_mu, config_trk_end_x]
import matplotlib.pyplot as plt
from matplotlib.backends.backend_agg import FigureCanvasAgg
from IPython.display import display
import numpy as np

for var_config in var_configs:
    n_masks = len(mask_dict)

    original_show = plt.show
    plt.show = lambda: None

    figs = []
    for name, mask_func in mask_dict.items():
        fig, red_chi2 = plot_stacked_histogram_with_ratio(
            mc_df=mc_evt_df[mask_func(mc_evt_df)],
            data_df=data_evt_df[mask_func(data_evt_df)],
            config=var_config,
            cov_frac_matrix=None, cov_matrix=None,
            title="", weight_column=pot_weight_col,
            data_pot=data_tot_pot, normalize=False,
            show_stats=False, symmetric_ratio=True, divide_by_bin_width=False
        )
        figs.append((fig, name))

    plt.show = original_show

    n_cols = min(3, n_masks)
    n_rows = int(np.ceil(n_masks / n_cols))

    grid_fig = plt.figure(figsize=(10 * n_cols, 8 * n_rows))

    for idx, (sub_fig, name) in enumerate(figs):
        canvas = FigureCanvasAgg(sub_fig)
        canvas.draw()
        img = np.asarray(canvas.buffer_rgba())

        ax = grid_fig.add_subplot(n_rows, n_cols, idx + 1)
        ax.imshow(img)
        ax.set_title(name, fontsize=14, fontweight='bold', pad=8)
        ax.axis('off')
        plt.close(sub_fig)

    grid_fig.suptitle(var_config.xlabel, fontsize=16, fontweight='bold', y=1.01)
    grid_fig.tight_layout()
    display(grid_fig)       # <-- replaces plt.show()
    plt.close(grid_fig)


mask_dict = {
    "TPC0": lambda df: (df.slc.vertex.x < 0),
    "TPC1": lambda df: (df.slc.vertex.x > 0),
    "TPC_contained": lambda df: TPC_containment_mask(df,group_levels),
}

var_configs = [config_phi_mu, config_trk_end_x]
import matplotlib.pyplot as plt
from matplotlib.backends.backend_agg import FigureCanvasAgg
from IPython.display import display
import numpy as np

for var_config in var_configs:
    n_masks = len(mask_dict)

    original_show = plt.show
    plt.show = lambda: None

    figs = []
    for name, mask_func in mask_dict.items():
        fig, red_chi2 = plot_stacked_histogram_with_ratio(
            mc_df=mc_evt_df[mask_func(mc_evt_df)],
            data_df=data_evt_df[mask_func(data_evt_df)],
            config=var_config,
            cov_frac_matrix=None, cov_matrix=None,
            title="", weight_column=pot_weight_col,
            data_pot=data_tot_pot, normalize=False,
            show_stats=False, symmetric_ratio=True, divide_by_bin_width=False
        )
        figs.append((fig, name))

    plt.show = original_show

    n_cols = min(3, n_masks)
    n_rows = int(np.ceil(n_masks / n_cols))

    grid_fig = plt.figure(figsize=(10 * n_cols, 8 * n_rows))

    for idx, (sub_fig, name) in enumerate(figs):
        canvas = FigureCanvasAgg(sub_fig)
        canvas.draw()
        img = np.asarray(canvas.buffer_rgba())

        ax = grid_fig.add_subplot(n_rows, n_cols, idx + 1)
        ax.imshow(img)
        ax.set_title(name, fontsize=14, fontweight='bold', pad=8)
        ax.axis('off')
        plt.close(sub_fig)

    grid_fig.suptitle(var_config.xlabel, fontsize=16, fontweight='bold', y=1.01)
    grid_fig.tight_layout()
    display(grid_fig)       # <-- replaces plt.show()
    plt.close(grid_fig)